# EP05｜1000 個 AI agent 會快 1000 倍嗎？

像素腦底加！這是影片裡的模擬程式。按上方的 **執行階段 → 全部執行**（或每格按 ▶️）就能自己跑一遍，幾秒鐘就跑完，不用顯示卡。

⚠️ 這是把問題簡化過的模擬：工作時間是隨機產生的，agent 之間溝通不花時間，目的是看「工作互相等待」會造成什麼結果，不是在重現任何一家公司的真實系統。

📺 影片：[youtube.com/@pixelbraintw](https://www.youtube.com/@pixelbraintw)

## 0. 準備

In [ ]:
import heapq
import json
import random

random.seed(5)

## 1. 做一個假的大專案：2000 個工作、40 層

In [ ]:
# 1) 做一個假的大專案：2000 個工作，分成 40 層
#    每個工作花 0.5 到 3 小時，而且要等上一層的 3 個工作做完才能開始
N, LAYERS = 2000, 40
tasks = []
per = N // LAYERS
for i in range(N):
    L = i // per
    need = random.sample(range((L - 1) * per, L * per), 3) if L > 0 else []
    tasks.append({"hours": round(random.uniform(0.5, 3.0), 2), "needs": need})
total = sum(t["hours"] for t in tasks)
print(f"{N} 個工作，全部加起來 {total:.1f} 小時")

## 2. 關鍵路徑：最長的那條「一定要依序做」的工作鏈

In [ ]:
# 2) 最長的那條「一定要依序做」的工作鏈（關鍵路徑）
finish = [0.0] * N
for i, t in enumerate(tasks):
    finish[i] = max([finish[j] for j in t["needs"]], default=0) + t["hours"]
critical = max(finish)


def graph():
    """每個工作還在等幾個前置工作、做完後會放行哪些工作、現在可以開始的工作。"""
    waiting = {i: len(t["needs"]) for i, t in enumerate(tasks)}
    children = [[] for _ in range(N)]
    for i, t in enumerate(tasks):
        for j in t["needs"]:
            children[j].append(i)
    ready = [i for i in range(N) if waiting[i] == 0]
    heapq.heapify(ready)
    return waiting, children, ready

## 3. 派給 k 個 agent：從 1 個加到 1024 個
人多 1000 倍，會快 1000 倍嗎？

In [ ]:
# 3) 派給 k 個 agent：誰有空，就接一個前面的工作都做完的工作
def run(k):
    waiting, children, ready = graph()
    free, now, busy, log = k, 0.0, [], []
    while ready or busy:
        while free and ready:
            i = heapq.heappop(ready)
            heapq.heappush(busy, (now + tasks[i]["hours"], i))
            free -= 1
            log.append((i, now, now + tasks[i]["hours"]))
        now, i = heapq.heappop(busy)
        free += 1
        for c in children[i]:
            waiting[c] -= 1
            if waiting[c] == 0:
                heapq.heappush(ready, c)
    return now, log


res = []
for k in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]:
    t, _ = run(k)
    print(f"{k:5d} 個 agent：{t:8.1f} 小時，快 {total / t:6.1f} 倍")
    res.append({"agents": k, "hours": round(t, 1), "speedup": round(total / t, 1)})
print(f"最長工作鏈 {critical:.1f} 小時；理論上最多快 {total / critical:.1f} 倍")

## 4. 故意做錯：拿掉測試
每個工作 1% 出錯，2000 個全對的機率是 0.99 的 2000 次方。

In [ ]:
# 4) 故意做錯：每個工作有 1% 機率出錯，沒有測試的話，整個專案全對的機率？
p = 0.01
formula_no = (1 - p) ** N
no_test = sum(all(random.random() > p for _ in range(N)) for _ in range(200))
# 有測試：99% 的錯會被抓到並重做，只剩 0.01% 的錯漏掉
miss = p * 0.01
with_test = (1 - miss) ** N
sim_with = sum(all(random.random() > miss for _ in range(N)) for _ in range(2000)) / 2000
print(f"沒有測試：全對機率 {formula_no:.2e}（模擬 200 次，全對 {no_test} 次）")
print(f"有測試：全對機率 {with_test:.3f}（模擬 2000 次，全對 {sim_with:.3f}）")

## 5. 1024 個 agent 的時候，到底有幾個在做事？

In [ ]:
# 5) 1024 個 agent 的時候，每個時間點有幾個在做事？
_, log = run(1024)
end = max(e for _, _, e in log)
busy_n = [sum(1 for _, a, e in log if a <= h < e) for h in [x * 0.5 for x in range(int(end * 2) + 1)]]
print(f"1024 個 agent：同一時間最多 {max(busy_n)} 個在做事，平均 {sum(busy_n) / len(busy_n):.1f} 個")

## 6. 主 agent 變成瓶頸
每個工作做完都要先檢查 6 分鐘。只有 1 個負責檢查會怎樣？

In [ ]:
# 6) 主 agent 變成瓶頸：每個工作做完，要先給負責檢查的 agent 看 6 分鐘（0.1 小時），
#    通過了，後面的工作才能開始
def run_review(k, reviewers, review_hours=0.1):
    waiting, children, ready = graph()
    events, queue = [], []          # events: (時間, 種類, 工作)；queue: 排隊等檢查的工作
    free, free_r, now, maxq = k, reviewers, 0.0, 0
    while ready or events or queue:
        while free and ready:
            i = heapq.heappop(ready)
            heapq.heappush(events, (now + tasks[i]["hours"], 0, i))
            free -= 1
        while free_r and queue:
            i = queue.pop(0)
            heapq.heappush(events, (now + review_hours, 1, i))
            free_r -= 1
        maxq = max(maxq, len(queue))
        now, kind, i = heapq.heappop(events)
        if kind == 0:               # 做完了，去排隊等檢查
            free += 1
            queue.append(i)
        else:                       # 檢查通過，後面的工作可以開始
            free_r += 1
            for c in children[i]:
                waiting[c] -= 1
                if waiting[c] == 0:
                    heapq.heappush(ready, c)
    return now, maxq


rev = []
for r in [1, 2, 4, 8]:
    t, q = run_review(1024, r)
    print(f"1024 個 agent、{r} 個負責檢查：{t:6.1f} 小時，最多 {q} 個工作在排隊等檢查")
    rev.append({"reviewers": r, "hours": round(t, 1), "speedup": round(total / t, 1), "max_queue": q})

## 7. 換一種拆法
一樣 2000 個工作，拆成 10 層或 200 層，都給 1024 個 agent。

In [ ]:
# 7) 換一種拆法：一樣 2000 個工作，拆成 10 層（很寬）或 200 層（很深），都給 1024 個 agent
shape = []
orig_tasks = tasks
for layers in [10, 200]:
    rng = random.Random(100 + layers)
    per2 = N // layers
    tasks = []
    for i in range(N):
        L = i // per2
        need = rng.sample(range((L - 1) * per2, L * per2), 3) if L > 0 else []
        tasks.append({"hours": round(rng.uniform(0.5, 3.0), 2), "needs": need})
    tot = sum(t["hours"] for t in tasks)
    t, _ = run(1024)
    print(f"拆成 {layers:3d} 層（每層 {per2:3d} 個）：{t:6.1f} 小時，快 {tot / t:5.1f} 倍")
    shape.append({"layers": layers, "per_layer": per2, "total_hours": round(tot, 1), "hours": round(t, 1), "speedup": round(tot / t, 1)})
tasks = orig_tasks

## 8. 畫成圖：agent 數量和快幾倍

In [ ]:
import matplotlib.pyplot as plt

ks = [r["agents"] for r in res]
plt.figure(figsize=(7, 5))
plt.plot(ks, ks, "--", color="gray", label="ideal")
plt.plot(ks, [r["speedup"] for r in res], "o-", color="orange", label="simulation")
plt.xscale("log", base=2); plt.yscale("log", base=2)
plt.xlabel("number of agents"); plt.ylabel("speedup"); plt.legend(); plt.grid(alpha=.3)
plt.show()

## 9. 換你試試看
改下面的數字再執行：層數越少（越寬），能同時做的工作越多；層數越多（越深），大家越常在等。
你能找到比 128 倍更快的拆法嗎？

In [ ]:
LAYERS_TRY, AGENTS_TRY = 20, 1024   # ← 改這兩個數字

rng = random.Random(1)
per2 = N // LAYERS_TRY
orig_tasks, tasks = tasks, []
for i in range(N):
    L = i // per2
    need = rng.sample(range((L - 1) * per2, L * per2), min(3, per2)) if L > 0 else []
    tasks.append({"hours": round(rng.uniform(0.5, 3.0), 2), "needs": need})
tot = sum(t["hours"] for t in tasks)
t, _ = run(AGENTS_TRY)
print(f"拆成 {LAYERS_TRY} 層、{AGENTS_TRY} 個 agent：{t:.1f} 小時，快 {tot / t:.1f} 倍")
tasks = orig_tasks